In [20]:
from collections import defaultdict
import seaborn
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

In [21]:
env = gym.make("Blackjack-v1", render_mode="rgb_array", sab=True)

In [22]:
done = False
observation, info = env.reset()

In [24]:
action = env.action_space.sample()
observation, reward, terminated, truncated, info = env.step(action)

In [25]:
class BlackjackAgent():
    def __init__(self, lr:float, ep_initial:float, ep_decay:float, ep_final:float, discount_factor:float=0.95):
        self.q_values = defaultdict(lambda:np.zeros(env.action_space.n))
        self.lr = lr
        self.ep_initial = ep_initial
        self.ep_decay = ep_decay
        self.ep_final = ep_final
        self.df = discount_factor
        self.training_error = []

    def get_action(self, obs:tuple[int,int,bool])->int:
        if np.random.random()<self.ep_initial:
            return env.action_space.sample()
        else:
            return int(np.argmax(self.q_values[obs]))

    def update(self, obs, action, reward, terminated, next_obs):
        future_q_values = (not terminated) * np.max(self.q_values[next_obs])
        temp_diff = reward + self.df * future_q_values - self.q_values[obs][action]
        self.q_values[obs][action] += self.lr * temp_diff
        self.training_error.append(temp_diff)

    def decay_epsilon(self):
        self.ep_initial = max(self.ep_final, self.ep_initial-self.ep_decay)
        

In [26]:
lr = 0.01
n_ep = 1000
start_ep = 1.0
ep_decay = start_ep/(n_ep/2)
final_ep = 0.1
agent = BlackjackAgent(lr = lr, ep_initial=start_ep, ep_decay=ep_decay, ep_final=final_ep)

In [ ]:
from IPython.display import clear_output
env = gym.wrappers.RecordEpisodeStatistics(env,deque_size=n_ep)
for ep in tqdm(range(n_ep)):
    obs, info = env.reset()
    done = False
    clear_output()
    while not done:
        action = agent.get_action(obs)
        next_obs, reward, terminated, truncated, info = env.step(action)
        agent.update(obs,action,reward,terminated,next_obs)
        frame = env.render()
        plt.imshow(frame)
        plt.show()
        done = terminated or truncated
        obs = next_obs
agent.ep_decay